# Notebook 03B — T09: Polynomial và Ridge

Phần B riêng, tránh chỉnh cùng notebook mô hình của A. Dùng raw `realSum`, semantic allowlist và manifest T07. Chọn bậc/alpha bằng 5-fold CV trong train. **Không dự đoán hay đánh giá test ở notebook này**. Chưa đủ M0–M2 của A để kết luận thắng chung/T10.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (ROOT / 'src' / 'experiment.py').exists(), 'Mở notebook từ root hoặc notebooks/'
sys.path.insert(0, str(ROOT))

from src.b_tasks import run_b_models
from src.experiment import load_protocol


## 1. So sánh các cấu hình

Polynomial bậc 2/3 trên capacity/bedrooms/dist với context category và numeric còn lại; không khai triển toàn bộ dummy. Ridge alpha theo dãy ví dụ Bài 7. PolynomialFeatures/scaler/encoder nằm trong pipeline, được fit lại trong từng fold. Allowlist không được lọc theo kết quả Pearson/ANOVA.

`cv_*_mean/std` là trung bình/độ phân tán giữa folds, `fold_train_*` là điểm trên phần train mỗi fold. RMSE mean giữa folds khác sqrt(MSE mean). Tối ưu theo `cv_mse_mean`. Đây là CV dùng lựa chọn cấu hình, không ước lượng test cuối độc lập.

In [2]:
summary = run_b_models(ROOT)
display(summary[['model_id','variant','cv_mse_mean','cv_mse_std','cv_rmse_mean',
                 'cv_mae_mean','cv_r2_mean','fold_train_r2_mean','selected_within_family']])


M3 degree_2 CV MSE 79616.75


M3 degree_3 CV MSE 79228.0


M4 alpha_0.001 CV MSE 79988.15


M4 alpha_0.01 CV MSE 79988.15


M4 alpha_0.1 CV MSE 79988.15


M4 alpha_1 CV MSE 79988.33


M4 alpha_10 CV MSE 80006.64


M4 alpha_100 CV MSE 80725.81


M4 alpha_1000 CV MSE 84147.42


,model_id,variant,cv_mse_mean,cv_mse_std,cv_rmse_mean,cv_mae_mean,cv_r2_mean,fold_train_r2_mean,selected_within_family
0,M3,degree_2,79616.753843,18821.273541,280.495547,103.259941,0.244555,0.239354,False
1,M3,degree_3,79227.997866,18787.138667,279.800136,102.940256,0.248306,0.244840,True
2,M4,alpha_0.001,79988.148545,18800.210910,281.168391,104.351169,0.240876,0.234968,False
3,M4,alpha_0.01,79988.148290,18800.213259,281.168390,104.350935,0.240876,0.234968,False
4,M4,alpha_0.1,79988.147660,18800.236766,281.168385,104.348604,0.240876,0.234968,True
5,M4,alpha_1,79988.330639,18800.473735,281.168674,104.325647,0.240874,0.234966,False
6,M4,alpha_10,80006.636388,18802.982299,281.201203,104.132864,0.240697,0.234788,False
7,M4,alpha_100,80725.811686,18824.186459,282.489159,104.246179,0.233652,0.227875,False
8,M4,alpha_1000,84147.416327,18809.143704,288.553544,108.443160,0.199985,0.194969,False


## 2. Kết quả và giới hạn

Model tốt nhất trong mỗi family fit lại **chỉ train** để lưu. Đây chưa phải model tốt nhất chung. A tiếp tục M0/M1/M2 theo cùng protocol, rồi A+B chốt mô hình bằng CV trước test T10. Không tự lấy holdout để điều chỉnh bậc, alpha, predictor, clipping hoặc target.

In [3]:
display(Markdown((ROOT/'reports/b_model_findings.md').read_text(encoding='utf-8')))
folds = pd.read_csv(ROOT/'outputs/tables/b_models_cv_folds.csv')
assert len(folds)==45 and len(summary)==9
assert summary['selected_within_family'].sum()==2
assert not any('test' in col for col in summary.columns)
print('T09 hoàn thành train/CV; chưa đánh giá test, chờ T08 của A cho T10.')


# T09: đa thức và Ridge của B

Đã chạy train/CV, không đánh giá hoặc dự đoán test. So sánh cùng config, manifest và folds trong T07.

- M3: chọn degree_3 theo CV MSE trung bình 79228.00, fit lại chỉ train để lưu model.
- M4: chọn alpha_0.1 theo CV MSE trung bình 79988.15, fit lại chỉ train để lưu model.

Chưa tuyên bố mô hình thắng chung: phải nhận M0/M1/M2 của A và chốt bằng CV trước T10. Mean RMSE của folds khác sqrt(mean MSE); CSV lưu rõ từng fold và mean/std.

Allowlist cố định theo metadata trước CV; không lọc predictor theo target. Bậc 2/3 chỉ trên capacity, bedrooms, dist; các nhóm city/day/room và phần numeric còn lại dùng cùng biểu diễn nền. Ridge dùng dãy alpha đã có trong kế hoạch.


T09 hoàn thành train/CV; chưa đánh giá test, chờ T08 của A cho T10.
